# Neo RX V1.3.8 — entrenamiento completo en Kaggle

Este cuaderno ejecuta desde cero la auditoría, la separación de pacientes, el fine tuning completo, la evaluación y la exportación de artefactos.

1. Adjunta el dataset público **NIH Chest X-rays**.
2. Activa Internet y selecciona un acelerador GPU.
3. Ejecuta todo con **Run All** o **Save Version → Save & Run All**.
4. No cierres ni detengas la sesión hasta que termine la exportación.
5. No añadas secretos, datos clínicos ni archivos `.env`.


In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys, urllib.request

REPO_URL = "https://github.com/LinoMMJ/NEO-RX.git"
GIT_REF = "v1.3.8"
ROOT = Path("/kaggle/working/neorx")

try:
    with urllib.request.urlopen("https://github.com", timeout=15):
        pass
except Exception as exc:
    raise RuntimeError("Kaggle no tiene acceso a Internet. Actívalo en Settings antes de continuar.") from exc

if ROOT.exists() and not (ROOT / ".git").exists():
    shutil.rmtree(ROOT)

if not ROOT.exists():
    subprocess.run(["git", "clone", "--depth", "1", "--branch", GIT_REF, REPO_URL, str(ROOT)], check=True)
else:
    subprocess.run(["git", "fetch", "--tags", "origin"], cwd=ROOT, check=True)
    subprocess.run(["git", "checkout", "--force", GIT_REF], cwd=ROOT, check=True)

BACKEND = ROOT / "backend"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", str(BACKEND / "training/requirements.kaggle.txt")], check=True)
os.chdir(BACKEND)
sys.path.insert(0, str(BACKEND))
commit = subprocess.run(["git", "rev-parse", "--short", "HEAD"], cwd=ROOT, check=True, capture_output=True, text=True).stdout.strip()
print("Código:", ROOT)
print("Versión:", GIT_REF, commit)


In [ ]:
import torch, platform
print("Python:", platform.python_version())
print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError("Activa Accelerator > GPU antes del piloto o entrenamiento")
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM GB:", round(torch.cuda.get_device_properties(0).total_memory / 1024**3, 1))


In [ ]:
from pathlib import Path
import shutil
import yaml

input_candidates = []
for child in Path("/kaggle/input").iterdir():
    if any(child.rglob("Data_Entry_2017.csv")):
        input_candidates.append(child)
if len(input_candidates) != 1:
    raise RuntimeError(f"Se esperaba exactamente un dataset NIH adjunto; encontrados: {input_candidates}")

DATA_ROOT = input_candidates[0]
WORK = Path("/kaggle/working")
MANIFESTS = WORK / "manifests"
CHECKPOINTS = WORK / "checkpoints"
RESULTS = WORK / "results"
RUNS = WORK / "runs"

# Déjalo en True para comenzar el full sin checkpoints ni métricas de pruebas anteriores.
RESET_TRAINING_OUTPUTS = True
if RESET_TRAINING_OUTPUTS:
    for directory in (CHECKPOINTS, RESULTS, RUNS):
        shutil.rmtree(directory, ignore_errors=True)

for directory in (MANIFESTS, CHECKPOINTS, RESULTS, RUNS):
    directory.mkdir(parents=True, exist_ok=True)

with open(BACKEND / "training/config.kaggle.yaml", encoding="utf-8") as handle:
    config = yaml.safe_load(handle)
config["data"]["data_dir"] = str(DATA_ROOT)
config["data"]["manifest"] = str(MANIFESTS / "clean_manifest.csv")
config["data"]["splits_json"] = str(MANIFESTS / "splits.json")
config["model"]["checkpoint_dir"] = str(CHECKPOINTS)
config["train"]["log_dir"] = str(RUNS)
RUNTIME_CONFIG = WORK / "config.kaggle.runtime.yaml"
with open(RUNTIME_CONFIG, "w", encoding="utf-8") as handle:
    yaml.safe_dump(config, handle, allow_unicode=True, sort_keys=False)
print("Dataset:", DATA_ROOT)
print("Configuración:", RUNTIME_CONFIG)
print("Salidas anteriores eliminadas:", RESET_TRAINING_OUTPUTS)


## Auditoría

La auditoría lee el dataset directamente desde /kaggle/input; no copia los 42 GB a /kaggle/working. Para una comprobación inicial puedes añadir --max-images 2000, pero elimínalo antes del entrenamiento final.


In [ ]:
audit_command = [
    sys.executable, "-m", "training.validate_dataset",
    "--input-root", str(DATA_ROOT),
    "--output-dir", str(MANIFESTS),
    "--workers", "4",
]
if not (MANIFESTS / "clean_manifest.csv").exists():
    subprocess.run(audit_command, check=True)
else:
    print("Se reutiliza el manifiesto existente.")


In [ ]:
subprocess.run([
    sys.executable, "-m", "training.create_splits",
    "--manifest", str(MANIFESTS / "clean_manifest.csv"),
    "--output-dir", str(MANIFESTS),
    "--labels", "pulmonary",
    "--seed", "42",
], check=True)


## Entrenamiento completo

Esta ejecución usa todo el conjunto auditado. Los checkpoints se guardan después de cada época y el entrenamiento puede tardar varias horas.


In [ ]:
MODE = "full"
RESUME = None  # Usa la ruta de last.pt solamente para reanudar una ejecución interrumpida.

train_command = [
    sys.executable, "-m", "training.train",
    "--config", str(RUNTIME_CONFIG),
    "--device", "cuda",
]
if RESUME:
    train_command += ["--resume", RESUME]
print("Ejecutando:", " ".join(train_command))
subprocess.run(train_command, check=True)


## Evaluación final

Esta celda evalúa automáticamente el mejor modelo generado por el entrenamiento completo.


In [ ]:
FINAL = CHECKPOINTS / "finetuned_resnet50_nih.pt"
if MODE != "full":
    print("Evaluación omitida: el piloto no produce métricas finales válidas.")
elif not FINAL.exists():
    raise FileNotFoundError(FINAL)
else:
    subprocess.run([
        sys.executable, "-m", "training.evaluate",
        "--config", str(RUNTIME_CONFIG),
        "--checkpoint", str(FINAL),
        "--device", "cuda",
        "--output-dir", str(RESULTS),
        "--plot",
    ], check=True)


In [ ]:
import shutil

EXPORT = WORK / "neorx-v1.3.8-artifacts"
shutil.rmtree(EXPORT, ignore_errors=True)
EXPORT.mkdir(parents=True, exist_ok=True)

for directory in (CHECKPOINTS, MANIFESTS, RESULTS, RUNS):
    if directory.exists():
        shutil.copytree(directory, EXPORT / directory.name)
shutil.copy2(RUNTIME_CONFIG, EXPORT / RUNTIME_CONFIG.name)

archive = shutil.make_archive(str(EXPORT), "zip", EXPORT.parent, EXPORT.name)
print("Entrenamiento y evaluación completados.")
print("Descarga este archivo desde Output:", archive)
